# 🏁 Tier 1: The Foundation

These tasks ensure everyone is
* properly receiving and handling the data stream
* able to visualize both single frames and the (recorded) stream
* can do basic transformations

## Task 1: "Hello, Pointcloud!" Viewer
* **Goal:** Connect to the WebSocket server with rerun flag, to see the pointcloud stream. Save N frames to local machine and visualize them in different ways.
* **Why:** The fundamental first step. It always helps to *see* the data before starting to work with it.
* **Tools:** A basic (3D) visualization libraries, like pyplot, open3d, plotly and **Rerun**

### Task 1.1: Connect to the stream

`INSTRUCTOR_IP`, `PORT` and `TOKEN` are provided by the workshop organizer, fill them in and run:

```bash
# With Rerun visualization (recommended - best quality)
uv run scripts/exercise_client.py --server ws://INSTRUCTOR_IP:PORT --token TOKEN --rerun

# OR:
# If rerun is not working, try with matplotlib visualization
uv run scripts/exercise_client.py --server ws://INSTRUCTOR_IP:PORT --token TOKEN --visualize
```

NOTE 1: Scripts can be run from the terminal (recommended) or in notebook cells (example below)

NOTE 2: two lidars in two different IP:s and PORTS, you can try both!

Rerun (or matplotlib) window should pop up with pointcloud live stream. Stop with `ctrl + c` (terminal) or stipping the cell (notebook).

In [ ]:
# Option: Run commands in notebook cells like this:
# !uv run scripts/exercise_client.py --server ws://<IP>:<PORT> --token <TOKEN> --rerun

### Task 1.2: Record frames

Make a short recording of the stream:

```bash
# Save data to file
uv run scripts/exercise_client.py --server ws://INSTRUCTOR_IP:PORT --token TOKEN --save lidar_data.npz --nframes 200
```

### Task 1.3: Read the frames 

In [ ]:
# Read data from saved file
import numpy as np
filepath = 'lidar_data.npz'  # Adjust path if needed
data = np.load(filepath, allow_pickle=True)
frames = data['frames']
# See how many frames were loaded
len(frames)

In [ ]:
# Print a sample (one frame) to inspect its structure
frame_idx = # TODO: choose an index to inspect
test_frame = frames[frame_idx]
print(test_frame)
print(test_frame.keys())

In [ ]:
# Select points and intensity of the frame
test_points = test_frame['points']
test_intensity = test_frame['intensity']
# Print shape of points and intensity arrays
print(test_points.shape)
print(test_intensity.shape)

### Task 1.4: Try out different visualization libraries

TIP: You can test these with both lidars by making two recordings and changing between them in the previous steps.

In [ ]:
# Matplotlib: Visualize one frame
import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
scat = ax.scatter(
    test_points[:,0], # X coordinates
    test_points[:,1], # Y coordinates
    test_points[:,2], # Z coordinates
    c=test_intensity, 
    cmap='viridis', 
    s=1)
plt.colorbar(scat, label='Intensity')
plt.show()

In [ ]:
# Open3D: Visualize one frame (interactive, try to turn the pointcloud around!)
import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(test_points)

# OPTIONAL STEPS, uncomment if you wish to try them out:
# Transformation examples: Swap y..z and flip y
# pcd.points = o3d.utility.Vector3dVector(np.column_stack((test_points[:, 0], test_points[:, 2], -test_points[:, 1])))

# Map intensity to colors
# colors = np.zeros((test_points.shape[0], 3))
# colors[:, 0] = test_intensity / np.max(test_intensity)  # Red channel

# Other option: Distance from origin as colors
# distances = np.linalg.norm(test_points, axis=1)
# colors = np.zeros((test_points.shape[0], 3))
# colors[:, 1] = distances / np.max(distances)  # Green channel

# pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])


In [ ]:
# Plotly: Visualize one frame, interactive 3D plot
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(
    x= TODO,
    y= TODO,
    z= TODO,
    mode='markers',
    marker=dict(
        size=1,
        # color=test_intensity, # Comment out if you want to test
        # colorscale='Viridis',
        # colorbar=dict(title='Intensity'),
        opacity=0.8
    )
)])
fig.show()

In [ ]:
# Rerun: Visualize all recorded frames in for loop
# NOTE: Don't close the Rerun viewer window!
# That will cause some errors that require you to restart the kernel.
# To stop the loop, just stop this cell execution.
import rerun as rr
rr.init("lidar_lab_basics", spawn=True)

for i, frame in enumerate(frames):
    print(i, end='\r')
    pts = frame['points']
    intensity = frame['intensity']

    rr.log(
        "lidar/points",
        rr.Points3D(
            pts,
            colors=intensity,
            radii=0.5 # Adjust if needed
        )
    )


## Task 2: Colors and transformations
* **Goal:** Try different point coloring options, and learn to perform cloud transformations.
* **Why:** Colors help to see the pointcloud details! And usually they reveal some need for transformations, like turning the pointcloud upside down or filtering out some points
* **Tools:** Rerun (you can also try the coloring in other tools, but we'll focus on rerun from now on)

### Tasks:

In the cell below, try these things:
* **2.1**. Run rerun with intensity as color
* **2.2**. Figure out how to set *distance* from origin as color
* **2.3**. Figure out how to set *height* as color
* **2.4**. Try swapping and flipping the axes as simple transformations
* **2.5**. Try limiting the pointcloud: Only render points that fall within a specific 3D bounding box (e.g., `x` from 1.0 to 2.0, `y` from -0.5 to 0.5). 
* **2.6**. Extra: how would you rotate the whole frame, x.ex. 45 degrees on Y axis?

#### Rerun tips:

* **NOTE:** Don't close the Rerun Viewer between runs, rather stop the notebook cell. Otherwise the whole kernel needs to be restarted.
* Turn "Show origin axes" and "Show bounding box" on: `Blueprint` -> `/` -> `View properties`
* If you feel like it, the same tasks can already be done in `scripts/utils.py` function `log_rerun` usig the live stream (we'll go there later anyways)


In [ ]:
# Visualize all frames with rerun
import matplotlib.pyplot as plt
import rerun as rr
rr.init("lidar_lab_basics", spawn=True)

for frame in frames:
    pts = frame['points']
    intensity = frame['intensity']
    # AXIS TRANSFORMATION

    # (ROTATION)

    # FILTERING

    # COLORING
    # Color by height
    # TODO
    # Color by distance from origin
    # TODO

    # colors = distances / np.max(distances)
    # colors = heights / np.max(heights)
    colors = intensity / np.max(intensity) # Normalize intensity

    # Map to RGB using matplotlib colormap
    cmap = plt.get_cmap('viridis')
    rgba = cmap(colors)
    colors = (rgba[:, :3] * 255).astype(np.uint8)

    rr.log(
        "lidar/points",
        rr.Points3D(
            pts,
            colors=colors,
            radii=0.1
        )
    )


## Task 3: Voxel Grid Downsampling
* **Goal:** Reduce the number of points for performance. For each frame, create a 3D grid (voxels) and only keep one "average" point per occupied voxel.
* **Why:** Live streams can be overwhelmingly dense. This is a standard and essential performance optimization technique.

In [ ]:
# Write a function for voxel grid downsampling. TIP: Use LLM to help you write it.
def voxel_grid_downsample(points, voxel_size):
    """
    Input:
        points: Nx3 numpy array of point coordinates
        voxel_size: float, size of the voxel grid
    Output:
        downsampled_points: Mx3 numpy array of downsampled point coordinates
    """
    # TODO
    return np.array(downsampled_points)

In [ ]:
# Test the voxel grid downsampling function
voxel_size = 0.1
print(f'Frame size before downsampling: {test_points.shape[0]} points')
downsampled_test_points = voxel_grid_downsample(test_points, voxel_size=voxel_size)
print(f'Frame size after downsampling: {downsampled_test_points.shape[0]} points')

In [ ]:
# OPTIONAL: Implement the voxel grid downsampling function to the rerun visualization loop above.

## Next

1. Pick your go-to LiDAR and the transformations, coloring and downsampling options that make the most sense. Add them to `log_rerun` function in `scripts/utils.py` to have them available in live stream.
2. Go to `02_advanced.ipynb` notebook for follow up instructions.